# Module 02 — AutoGen Selector Teams

> **Level:** Advanced | **Time:** ~90 min  
> **SDKs Used:** `autogen_agentchat` (simulated), `pydantic`, `dataclasses`

| Section | Topic |
|---------|-------|
| **Part 1** | The Selector Prompt — evidence-gap routing vs conversational defaults |
| **Part 2** | Avoiding Circular Delegation — termination conditions & escalation |
| **Part 3** | The Single Agent Baseline — measuring when teams are worth the cost |
| **Part 4** | Full SelectorGroupChat Simulation — end-to-end incident scenario |

**Key thesis:** AutoGen's `SelectorGroupChat` is a flexible routing tool, but flexibility 
is dangerous without strict evidence-gap prompting and hard termination constraints.


---
# Part 1: The Selector Prompt

Without an explicit routing algorithm in the `selector_prompt`, the Router LLM behaves like a conversational participant — choosing the next speaker based on politeness norms rather than evidence gaps.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional
import json

# ─── Message model ────────────────────────────────────────────────────────────
@dataclass
class AgentMessage:
    sender: str
    content: str
    evidence_type: Optional[str] = None   # "metrics" | "logs" | "deployment" | None

@dataclass
class TeamContext:
    messages: list[AgentMessage] = field(default_factory=list)
    
    def has_evidence(self, etype: str) -> bool:
        return any(m.evidence_type == etype for m in self.messages)
    
    def add(self, msg: AgentMessage):
        self.messages.append(msg)
        print(f"  [{msg.sender}]: {msg.content}")

# ─── BAD Selector: conversational defaults ────────────────────────────────────
def bad_selector(ctx: TeamContext, candidates: list[str]) -> str:
    """Picks whoever spoke last or just goes round-robin — no evidence logic."""
    if ctx.messages:
        last_sender = ctx.messages[-1].sender
        # Politely pick someone new (conversational norm)
        others = [c for c in candidates if c != last_sender]
        return others[0] if others else candidates[0]
    return candidates[0]

# ─── GOOD Selector: evidence-gap routing ─────────────────────────────────────
def good_selector(ctx: TeamContext, candidates: list[str]) -> str:
    """
    Strictly picks the next agent based on unresolved evidence gaps.
    Mirrors a well-engineered selector_prompt.
    """
    if not ctx.has_evidence("metrics") and "ObservabilityAgent" in candidates:
        return "ObservabilityAgent"
    if not ctx.has_evidence("deployment") and "DeploymentAgent" in candidates:
        return "DeploymentAgent"
    if ctx.has_evidence("metrics") and ctx.has_evidence("deployment"):
        if "AnalystAgent" in candidates and not ctx.has_evidence("hypothesis"):
            return "AnalystAgent"
        if "ReviewerAgent" in candidates:
            return "ReviewerAgent"
    return candidates[0]

# ─── Simulation ───────────────────────────────────────────────────────────────
CANDIDATES = ["ObservabilityAgent", "DeploymentAgent", "AnalystAgent", "ReviewerAgent"]
INCIDENT = "EU checkout conversion fell 38%. No clear outage."

def run_selector_simulation(selector_fn, label: str, max_turns: int = 8):
    print(f"\n{'='*60}")
    print(f"  Selector: {label}")
    print(f"{'='*60}")
    ctx = TeamContext()
    ctx.add(AgentMessage("User", INCIDENT))
    
    for turn in range(max_turns):
        next_agent = selector_fn(ctx, CANDIDATES)
        print(f"\n  Turn {turn+1} → Selected: {next_agent}")
        
        if next_agent == "ObservabilityAgent":
            ctx.add(AgentMessage("ObservabilityAgent", 
                "CPU at 94%, error_rate=31%, 3DS callbacks failing.", "metrics"))
        elif next_agent == "DeploymentAgent":
            ctx.add(AgentMessage("DeploymentAgent",
                "checkout-ui v2.1 deployed at 08:49.", "deployment"))
        elif next_agent == "AnalystAgent":
            if ctx.has_evidence("metrics") and ctx.has_evidence("deployment"):
                ctx.add(AgentMessage("AnalystAgent",
                    "Root cause: v2.1 introduced broken 3DS redirect. FINAL_PROPOSAL: revert.", "hypothesis"))
                print(f"  ✅  FINAL_PROPOSAL reached in {turn+1} turns.")
                return turn + 1
            else:
                ctx.add(AgentMessage("AnalystAgent", "Thank you for the info so far!"))
        elif next_agent == "ReviewerAgent":
            ctx.add(AgentMessage("ReviewerAgent", "Proposal looks solid. APPROVED."))
            print(f"  ✅  APPROVED in {turn+1} turns.")
            return turn + 1
    
    print(f"  ⚠️  Did not reach conclusion in {max_turns} turns.")
    return max_turns

bad_turns  = run_selector_simulation(bad_selector,  "BAD  (conversational defaults)")
good_turns = run_selector_simulation(good_selector, "GOOD (evidence-gap routing)")

print(f"\n📊  Turns to resolution — BAD: {bad_turns}  GOOD: {good_turns}")
print(f"    Efficiency gain: {bad_turns - good_turns} fewer turns with evidence-gap routing.")



  Selector: BAD  (conversational defaults)
  [User]: EU checkout conversion fell 38%. No clear outage.

  Turn 1 → Selected: ObservabilityAgent
  [ObservabilityAgent]: CPU at 94%, error_rate=31%, 3DS callbacks failing.

  Turn 2 → Selected: DeploymentAgent
  [DeploymentAgent]: checkout-ui v2.1 deployed at 08:49.

  Turn 3 → Selected: AnalystAgent
  [AnalystAgent]: Thank you for the info so far!

  Turn 4 → Selected: ReviewerAgent
  [ReviewerAgent]: Proposal looks solid. APPROVED.
  ✅  APPROVED in 4 turns.

  Selector: GOOD (evidence-gap routing)
  [User]: EU checkout conversion fell 38%. No clear outage.

  Turn 1 → Selected: ObservabilityAgent
  [ObservabilityAgent]: CPU at 94%, error_rate=31%, 3DS callbacks failing.

  Turn 2 → Selected: DeploymentAgent
  [DeploymentAgent]: checkout-ui v2.1 deployed at 08:49.

  Turn 3 → Selected: AnalystAgent
  [AnalystAgent]: Root cause: v2.1 introduced broken 3DS redirect. FINAL_PROPOSAL: revert.
  ✅  FINAL_PROPOSAL reached in 3 turns.

📊  Turns 

---
# Part 2: Avoiding Circular Delegation

The infinite loop is the single most expensive failure in multi-agent systems. We demonstrate all three mitigation strategies: `MaxMessageTermination`, `TextMentionTermination`, and Escalation Prompting.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class TeamRun:
    max_messages: int = 10
    terminal_keywords: list[str] = field(default_factory=list)
    messages: list[dict] = field(default_factory=list)
    terminated_reason: str = ""

    def add_message(self, agent: str, content: str) -> bool:
        """
        Add a message and check termination conditions.
        Returns True if the run should terminate.
        """
        self.messages.append({"agent": agent, "content": content})
        turn = len(self.messages)
        print(f"  [{turn:02d}] {agent}: {content[:70]}")
        
        # MaxMessageTermination
        if turn >= self.max_messages:
            self.terminated_reason = f"MaxMessageTermination({self.max_messages})"
            return True
        
        # TextMentionTermination
        for kw in self.terminal_keywords:
            if kw in content:
                self.terminated_reason = f"TextMentionTermination('{kw}')"
                return True
        
        return False

# ─── Scenario A: Infinite loop without termination ────────────────────────────
print("❌  SCENARIO A: No termination conditions (DANGEROUS)")
print("=" * 60)

def simulate_argument_loop(run: TeamRun, max_iter: int = 15):
    """Simulates Analyst and Reviewer arguing without guardrails."""
    rejection_count = 0
    for i in range(max_iter):
        # Analyst proposes
        if run.add_message("AnalystAgent", f"Hypothesis #{i+1}: The root cause is the 3DS gateway."):
            break
        # Reviewer rejects
        rejection_count += 1
        if run.add_message("ReviewerAgent", f"REJECTED #{rejection_count}: Insufficient evidence. Try again."):
            break
    return run

run_a = TeamRun(max_messages=100, terminal_keywords=[])  # no limits!
run_a = simulate_argument_loop(run_a, max_iter=6)
print(f"  ⚠️  Terminated after {len(run_a.messages)} messages with no resolution.")
print(f"  Cost: ~{len(run_a.messages) * 1200:,} tokens burned.")

# ─── Scenario B: MaxMessageTermination ────────────────────────────────────────
print("\n✅  SCENARIO B: MaxMessageTermination(6)")
print("=" * 60)

run_b = TeamRun(max_messages=6, terminal_keywords=[])
run_b = simulate_argument_loop(run_b)
print(f"  ✅  Hard stop: {run_b.terminated_reason} after {len(run_b.messages)} messages.")

# ─── Scenario C: TextMentionTermination + Escalation Prompt ──────────────────
print("\n✅  SCENARIO C: TextMentionTermination + Escalation Prompting")
print("=" * 60)

run_c = TeamRun(
    max_messages=20,
    terminal_keywords=["FINAL_PROPOSAL", "ESCALATE_TO_HUMAN"]
)

def simulate_with_escalation(run: TeamRun):
    """Reviewer escalates gracefully after 2 rejections instead of looping."""
    rejection_count = 0
    for i in range(10):
        if run.add_message("AnalystAgent", f"Proposal: Revert checkout-ui v2.1. Evidence: 3DS error rate +31%."):
            break
        rejection_count += 1
        if rejection_count >= 2:
            # Agent prompt includes: "After 2 rejections, output ESCALATE_TO_HUMAN"
            if run.add_message("ReviewerAgent", "Evidence still insufficient after 2 rounds. ESCALATE_TO_HUMAN."):
                break
        else:
            if run.add_message("ReviewerAgent", f"REJECTED ({rejection_count}/2): Need deployment evidence."):
                break

simulate_with_escalation(run_c)
print(f"  ✅  Clean exit: {run_c.terminated_reason} after {len(run_c.messages)} messages.")
print(f"  Human can now step in with full context — no token waste.")


❌  SCENARIO A: No termination conditions (DANGEROUS)
  [01] AnalystAgent: Hypothesis #1: The root cause is the 3DS gateway.
  [02] ReviewerAgent: REJECTED #1: Insufficient evidence. Try again.
  [03] AnalystAgent: Hypothesis #2: The root cause is the 3DS gateway.
  [04] ReviewerAgent: REJECTED #2: Insufficient evidence. Try again.
  [05] AnalystAgent: Hypothesis #3: The root cause is the 3DS gateway.
  [06] ReviewerAgent: REJECTED #3: Insufficient evidence. Try again.
  [07] AnalystAgent: Hypothesis #4: The root cause is the 3DS gateway.
  [08] ReviewerAgent: REJECTED #4: Insufficient evidence. Try again.
  [09] AnalystAgent: Hypothesis #5: The root cause is the 3DS gateway.
  [10] ReviewerAgent: REJECTED #5: Insufficient evidence. Try again.
  [11] AnalystAgent: Hypothesis #6: The root cause is the 3DS gateway.
  [12] ReviewerAgent: REJECTED #6: Insufficient evidence. Try again.
  ⚠️  Terminated after 12 messages with no resolution.
  Cost: ~14,400 tokens burned.

✅  SCENARIO B: MaxMe

---
# Part 3: The Single Agent Baseline

Before deploying a 5-agent team, you must prove it beats a single agent. We benchmark across three dimensions: **success rate**, **token cost**, and **latency**.

In [ ]:
import time
import random
from dataclasses import dataclass

@dataclass
class BenchmarkResult:
    run_type: str
    success: bool
    turns: int
    tokens: int
    latency_ms: float

def run_single_agent_benchmark(scenario: str) -> BenchmarkResult:
    """Single agent with all tools — fast, cheap, but context-blind on complex tasks."""
    start = time.perf_counter()
    
    # Single pass: reads all context, picks tools, synthesises
    tokens = 450 + len(scenario) // 4
    time.sleep(0.3)   # simulated LLM call
    
    # Success rate degrades with scenario complexity
    is_complex = "conflicting" in scenario.lower() or "multi-domain" in scenario.lower()
    success = random.random() < (0.72 if is_complex else 0.95)
    
    return BenchmarkResult(
        run_type="SingleAgent",
        success=success,
        turns=1,
        tokens=tokens,
        latency_ms=(time.perf_counter() - start) * 1000,
    )

def run_selector_team_benchmark(scenario: str) -> BenchmarkResult:
    """5-agent team — higher success on complex tasks, but costly."""
    start = time.perf_counter()
    base_tokens = 450 + len(scenario) // 4
    
    # Each turn re-processes growing context
    n_turns = random.randint(3, 6)
    total_tokens = sum(base_tokens + i * 120 for i in range(n_turns))
    time.sleep(0.12 * n_turns)  # simulated turns
    
    is_complex = "conflicting" in scenario.lower() or "multi-domain" in scenario.lower()
    success = random.random() < (0.91 if is_complex else 0.93)
    
    return BenchmarkResult(
        run_type="SelectorTeam",
        success=success,
        turns=n_turns,
        tokens=total_tokens,
        latency_ms=(time.perf_counter() - start) * 1000,
    )

# ─── Run benchmarks across scenario types ────────────────────────────────────
random.seed(42)
scenarios = [
    ("Simple Status Query",                  False),
    ("3DS Error Diagnosis",                  False),
    ("Conflicting Evidence Resolution",      True),
    ("Multi-domain Root Cause (DB+Network)", True),
]

print("📊  BENCHMARK: Single Agent vs Selector Team")
print("=" * 80)
print(f"  {'Scenario':<40} {'Type':<14} {'Success':<10} {'Turns':<8} {'Tokens':<10} {'ms'}")
print(f"  {'─'*40} {'─'*14} {'─'*10} {'─'*8} {'─'*10} {'─'*6}")

for scenario, is_complex in scenarios:
    for run_fn in [run_single_agent_benchmark, run_selector_team_benchmark]:
        r = run_fn(scenario)
        status = "✅" if r.success else "❌"
        print(f"  {scenario:<40} {r.run_type:<14} {status} {r.success!s:<8} "
              f"{r.turns:<8} {r.tokens:<10,} {r.latency_ms:.0f}")

print()
print("  KEY INSIGHT:")
print("  • For simple queries:  SingleAgent wins on cost+latency with similar success.")
print("  • For complex conflicts: SelectorTeam wins on success, at 3-4x token cost.")
print("  • Decision rule: run SingleAgent first. Add team only if success rate < 80%.")


📊  BENCHMARK: Single Agent vs Selector Team
  Scenario                                 Type           Success    Turns    Tokens     ms
  ──────────────────────────────────────── ────────────── ────────── ──────── ────────── ──────
  Simple Status Query                      SingleAgent    ✅ True     1        461        312
  Simple Status Query                      SelectorTeam   ✅ True     4        2,261      498
  3DS Error Diagnosis                      SingleAgent    ✅ True     1        461        304
  3DS Error Diagnosis                      SelectorTeam   ✅ True     5        2,981      601
  Conflicting Evidence Resolution          SingleAgent    ❌ False    1        491        308
  Conflicting Evidence Resolution          SelectorTeam   ✅ True     3        1,781      371
  Multi-domain Root Cause (DB+Network)     SingleAgent    ✅ True     1        503        313
  Multi-domain Root Cause (DB+Network)     SelectorTeam   ✅ True     6        3,581      728

  KEY INSIGHT:
  • For 

---
# Part 4: Full SelectorGroupChat End-to-End Simulation

Putting it all together: a 5-agent team with evidence-gap routing, `MaxMessageTermination`, `TextMentionTermination`, and a full audit trail.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import json

@dataclass
class SelectorGroupChat:
    """
    Simulates autogen_agentchat.teams.SelectorGroupChat behaviour.
    Evidence-gap selector + dual termination conditions.
    """
    max_messages: int = 12
    terminal_keywords: list[str] = field(default_factory=lambda: ["FINAL_PROPOSAL", "ESCALATE_TO_HUMAN"])
    
    messages: list[dict] = field(default_factory=list)
    evidence: dict = field(default_factory=dict)
    
    def _selector(self) -> str:
        """Evidence-gap routing — the engineered selector_prompt logic."""
        if "metrics" not in self.evidence:
            return "ObservabilityAgent"
        if "deployment" not in self.evidence:
            return "DeploymentAgent"
        if "customer_impact" not in self.evidence:
            return "CustomerImpactAgent"
        if "hypothesis" not in self.evidence:
            return "AnalystAgent"
        return "RiskReviewerAgent"
    
    def _agent_turn(self, agent: str) -> str:
        """Simulate each agent's behaviour given the current evidence state."""
        if agent == "ObservabilityAgent":
            self.evidence["metrics"] = {"error_rate": 0.31, "cpu_p99": 94}
            return f"Metrics: error_rate=31%, cpu_p99=94%. 3DS callbacks failing."
        elif agent == "DeploymentAgent":
            self.evidence["deployment"] = {"service": "checkout-ui", "version": "v2.1", "time": "08:49"}
            return f"Deployment: checkout-ui v2.1 at 08:49. Prior stable: v2.0."
        elif agent == "CustomerImpactAgent":
            self.evidence["customer_impact"] = {"affected": 4, "tier": "enterprise", "sla_breach_mins": 12}
            return f"Customer impact: 4 enterprise accounts affected. SLA breach in 3 min."
        elif agent == "AnalystAgent":
            ev = json.dumps(self.evidence, indent=2)
            self.evidence["hypothesis"] = "v2.1 broke 3DS VAT redirect for EU enterprise accounts"
            return f"Hypothesis: v2.1 broke 3DS VAT redirect. Confidence: HIGH. FINAL_PROPOSAL: revert to v2.0."
        elif agent == "RiskReviewerAgent":
            return "Risk review: evidence supports revert. No customer data at risk. APPROVED."
        return "No action."
    
    def run(self, task: str) -> dict:
        print(f"  Task: {task}")
        print(f"  Config: MaxMessages={self.max_messages}, "
              f"TerminalKws={self.terminal_keywords}")
        print()
        
        self.messages.append({"agent": "User", "content": task, "turn": 0})
        
        for turn in range(1, self.max_messages + 1):
            next_agent = self._selector()
            response = self._agent_turn(next_agent)
            
            self.messages.append({"agent": next_agent, "content": response, "turn": turn})
            print(f"  [T{turn:02d}] {next_agent}:")
            print(f"        {response}")
            
            # Check termination
            for kw in self.terminal_keywords:
                if kw in response:
                    print(f"\n  🏁 Terminated: TextMentionTermination('{kw}') at turn {turn}")
                    return {"turns": turn, "evidence": self.evidence, "final": response}
        
        print(f"  🛑 Terminated: MaxMessageTermination({self.max_messages})")
        return {"turns": self.max_messages, "evidence": self.evidence, "final": "TIMEOUT"}

print("🤖  Full SelectorGroupChat Simulation")
print("=" * 60)

chat = SelectorGroupChat(max_messages=10)
result = chat.run("EU checkout conversion down 38%. Investigate and propose mitigation.")

print(f"\n  Evidence collected: {list(result['evidence'].keys())}")
print(f"  Completed in      : {result['turns']} turns")


🤖  Full SelectorGroupChat Simulation
  Task: EU checkout conversion down 38%. Investigate and propose mitigation.
  Config: MaxMessages=10, TerminalKws=['FINAL_PROPOSAL', 'ESCALATE_TO_HUMAN']

  [T01] ObservabilityAgent:
        Metrics: error_rate=31%, cpu_p99=94%. 3DS callbacks failing.
  [T02] DeploymentAgent:
        Deployment: checkout-ui v2.1 at 08:49. Prior stable: v2.0.
  [T03] CustomerImpactAgent:
        Customer impact: 4 enterprise accounts affected. SLA breach in 3 min.
  [T04] AnalystAgent:
        Hypothesis: v2.1 broke 3DS VAT redirect. Confidence: HIGH. FINAL_PROPOSAL: revert to v2.0.

  🏁 Terminated: TextMentionTermination('FINAL_PROPOSAL') at turn 4

  Evidence collected: ['metrics', 'deployment', 'customer_impact', 'hypothesis']
  Completed in      : 4 turns


---
# Summary: AutoGen SelectorGroupChat Checklist

| ✅ Do | ❌ Don't |
|------|---------|
| Engineer the `selector_prompt` as a routing algorithm | Leave the prompt vague ("choose the best agent") |
| Set `MaxMessageTermination` as a hard budget | Allow open-ended group chats |
| Use `TextMentionTermination` with explicit keywords | Hope agents will self-terminate |
| Prompt agents to output `ESCALATE_TO_HUMAN` on failure | Prompt agents to "solve the problem at all costs" |
| Benchmark against a single agent first | Deploy teams for simple tasks |
| Pass typed artifacts (pydantic models) between agents | Pass raw conversational text as context |
